# Medical-data model training

This notebook tunes MLP, GNN, and XGBoost models on the medical dataset using fixed cross-validation folds. It validates fold isolation and graph construction before running Optuna studies tracked with MLflow.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import gc
import logging
import os
import pickle
import random
import warnings

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from dotenv import load_dotenv
from lightning.pytorch.utilities.warnings import PossibleUserWarning
from sklearn import set_config

from my_project.data.utils import load_column_config
from my_project.experiments import medical as medical_cfg
from my_project.gnn.models import MyGNN, MyGNNConcat, MyMLP
from my_project.gnn.utils import build_graph_dataset, prepare_graph
from my_project.graphs import (
    build_base_graphs,
    build_ii_graphs,
    build_node_permutation,
    build_true_graph,
)
from my_project.information_theory import compute_info
from my_project.training.gnn_training import run_fixed_folds_sweep
from my_project.training.xgboost_training import run_xgboost_fixed_folds_study

warnings.filterwarnings(
    "ignore",
    message=r"`isinstance\(treespec, LeafSpec\)` is deprecated.*",
    module=r"lightning\.pytorch\.utilities\._pytree",
)

warnings.filterwarnings(
    "ignore",
    message=r".*'(train|val|test)_dataloader' does not have many workers.*",
    category=PossibleUserWarning,
)

load_dotenv(override=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

set_config(transform_output="pandas")
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.set_float32_matmul_precision("high")

for name in [
    "lightning",
    "lightning.pytorch",
    "lightning.fabric",
    "lightning.pytorch.utilities.rank_zero",
    "mlflow",
]:
    logging.getLogger(name).setLevel(logging.WARNING)

CUDA_AVAILABLE = torch.cuda.is_available()

In [ ]:
gc.collect()

if CUDA_AVAILABLE:
    torch.cuda.empty_cache()

## Configuration

Preprocessing pipelines and hyperparameter spaces use `my_project.experiments.medical` as their single source of truth, independently of the synthetic-data experiment configuration.

In [ ]:
GNN_SEARCH_SPACE = medical_cfg.GNN_SEARCH_SPACE
MLP_SEARCH_SPACE = medical_cfg.MLP_SEARCH_SPACE
XGB_SEARCH_SPACE = medical_cfg.XGB_SEARCH_SPACE

In [ ]:
EXPERIMENT_NAME = "medical"

In [ ]:
N_FOLDS = 5
selected_cols_key = "MODELING_FEATURES"

In [ ]:
MODEL_CONFIGS = [
    {
        "model_cls": MyMLP,
        "convs": [None],
        "search_space": MLP_SEARCH_SPACE,
    },
    {
        "model_cls": MyGNN,
        "convs": ["GATConv"],
        "search_space": GNN_SEARCH_SPACE,
    },
]

# build_base_graphs names the empty graph "empty", not "empty_graph".
MLP_REFERENCE_GRAPH = "empty"

# NN and XGBoost preprocessing pipelines from the medical experiment module.
PREPROCESSING_PIPELINE = copy.deepcopy(medical_cfg.NN_PREPROCESSING_PIPELINE)
XGBOOST_PREPROCESSING_PIPELINE = copy.deepcopy(
    medical_cfg.XGBOOST_PREPROCESSING_PIPELINE
)

EXPERIMENT_SETTINGS = {
    "seed": 42,
    "data_type": "medical",
    "n_trials": 50,
    "n_startup_trials": 10,
    "max_epochs": 200,
    "monitor_metric": "val/avg_precision",
    "monitor_mode": "max",
    "num_workers": 1,
    "optuna_n_jobs": 1,
    "experiment_name": EXPERIMENT_NAME,
    "pruning_mode": "fold",
    "pruning_warmup_steps": 1,
}

# Legacy variable aliases kept for compatibility with the rest of the notebook.
SEED = EXPERIMENT_SETTINGS["seed"]
DATA_TYPE = EXPERIMENT_SETTINGS["data_type"]
N_TRIALS = EXPERIMENT_SETTINGS["n_trials"]
N_STARTUP_TRIALS = EXPERIMENT_SETTINGS["n_startup_trials"]
MAX_EPOCHS = EXPERIMENT_SETTINGS["max_epochs"]
MONITOR_METRIC = EXPERIMENT_SETTINGS["monitor_metric"]
MONITOR_MODE = EXPERIMENT_SETTINGS["monitor_mode"]
DIRECTION = "minimize" if MONITOR_MODE == "min" else "maximize"
NUM_WORKERS = EXPERIMENT_SETTINGS["num_workers"]
OPTUNA_N_JOBS = EXPERIMENT_SETTINGS["optuna_n_jobs"]
EXPERIMENT_NAME = EXPERIMENT_SETTINGS["experiment_name"]
PRUNING_MODE = EXPERIMENT_SETTINGS["pruning_mode"]
PRUNING_WARMUP_STEPS = EXPERIMENT_SETTINGS["pruning_warmup_steps"]

PRECISION = "16-mixed" if CUDA_AVAILABLE else "32-true"

# Pruning mode passed to run_optuna_study_for_gnn:
#  - "epoch": prune after epochs (OptunaPruningCallback in train_gnn); step = epoch.
#  - "fold":  prune between folds; step = fold index. PercentilePruner counts
#             n_warmup_steps in folds, so it must be lower than the fold count.
#  - None:    disable pruning.

## Data and CV folds

In [ ]:
CONFIGS_DIR = os.getenv("CONFIGS_DIR")
DATA_DIR = os.getenv("DATA_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")
OPTUNA_STORAGE_URL = os.getenv("OPTUNA_STORAGE_URL")
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")

RESULT_DIR_TRAINING = os.path.join(RESULTS_DIR, "training", "gnn")

with open(os.path.join(CONFIGS_DIR, "data.yaml"), "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

col_cfg = load_column_config(CONFIGS_DIR)

base_cv_path = os.path.join(DATA_DIR, data_config["paths"]["crossval_indices_path"])
with open(base_cv_path, "rb") as f:
    base_folds = pickle.load(f)


print(len(base_folds), N_FOLDS)

if len(base_folds) < N_FOLDS:
    raise ValueError(f"Expected at least {N_FOLDS} CV folds, got {len(base_folds)}")
folds = list(base_folds)[:N_FOLDS]

In [ ]:
SELECTED_COLUMNS = getattr(col_cfg, selected_cols_key)

In [ ]:
df = pd.read_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["preprocessed_data_path"])
)

target_col = col_cfg.TARGET

y = df[target_col]
X = df[SELECTED_COLUMNS]

In [ ]:
# Fail fast before tuning if the persisted CV split violates isolation assumptions.
assert X.index.is_unique, "Dataset index must be unique for label-based fold selection."
assert X.index.equals(y.index), "X and y indices must be identical."
assert target_col not in X.columns, "Target column leaked into model features."
assert not X.columns.has_duplicates, "Model feature names must be unique."
assert len(folds) == N_FOLDS

dataset_index = set(X.index)
held_out_test = pd.Index(folds[0]["test"])
validation_counts = pd.Series(0, index=X.index, dtype="int64")
fold_audit_rows = []

for fold_idx, fold in enumerate(folds):
    train_index = pd.Index(fold["train"])
    valid_index = pd.Index(fold["valid"])
    test_index = pd.Index(fold["test"])

    assert train_index.intersection(valid_index).empty
    assert train_index.intersection(test_index).empty
    assert valid_index.intersection(test_index).empty
    assert test_index.equals(
        held_out_test
    ), "Held-out test membership changed across folds."
    assert set(train_index) | set(valid_index) | set(test_index) == dataset_index

    validation_counts.loc[valid_index] += 1
    fold_audit_rows.append(
        {
            "fold": fold_idx,
            "train_rows": len(train_index),
            "valid_rows": len(valid_index),
            "test_rows": len(test_index),
            "train_event_rate": y.loc[train_index].mean(),
            "valid_event_rate": y.loc[valid_index].mean(),
        }
    )

development_index = X.index.difference(held_out_test)
assert (validation_counts.loc[development_index] == 1).all()
assert (validation_counts.loc[held_out_test] == 0).all()

split_audit = pd.DataFrame(fold_audit_rows)
print("Split audit passed: disjoint folds, fixed held-out test, complete CV coverage.")
split_audit

In [ ]:
# Per-run MLflow tags describing the dataset. medical is REAL data, so the
# synthetic-only knobs (cov, generator_*, noise_level, ...) don't apply, but we
# still log the descriptors the evaluation notebook groups by (n_samples,
# train_size, valid_size, test_size, data_seed) so medical runs behave like the
# synthetic sweeps.
DATA_TAGS = {
    "data_type": DATA_TYPE,
    "generator_name": DATA_TYPE,
    "n_samples": len(X),
    "train_size": len(folds[0]["train"]),
    "valid_size": len(folds[0]["valid"]),
    "test_size": len(folds[0]["test"]),
    "data_seed": SEED,
}

## Per-fold graph builders

In [ ]:
INTERACTION_THRESHOLDS = [
    0.95,
    0.9,
    0.8,
]

include_empty_full = True

PERMUTE_SEEDS = list(range(0, 10))
# PERMUTE_SEEDS = list(range(0, 6))
# PERMUTE_SEEDS=[]

GRAPH_PREPROCESSING_PIPELINE = copy.deepcopy(medical_cfg.GRAPH_PREPROCESSING_PIPELINE)

ii_matrices = []
for fold in folds:
    X_train = X.loc[fold["train"]]
    y_train = y.loc[fold["train"]]
    pipe = copy.deepcopy(GRAPH_PREPROCESSING_PIPELINE)
    graph_input = pipe.fit_transform(X_train, y_train)
    ii_matrices.append(compute_info(graph_input, y_train, n_bins=None))

reference_columns = list(ii_matrices[0].columns)
if any(set(ii.columns) != set(reference_columns) for ii in ii_matrices[1:]):
    raise ValueError("Preprocessing produced different feature sets across folds.")
permutation_mappings = {
    seed: build_node_permutation(reference_columns, seed) for seed in PERMUTE_SEEDS
}

In [ ]:
fold = folds[0]

X_train = X.loc[fold["train"]]
y_train = y.loc[fold["train"]]

pipe = copy.deepcopy(GRAPH_PREPROCESSING_PIPELINE)
graph_input = pipe.fit_transform(X_train, y_train)

interaction = compute_info(graph_input, y_train, n_bins=None)

In [ ]:
graph_input.nunique().sort_values(ascending=False).head(
    10
), graph_input.nunique().sort_values(ascending=False).tail(10)

## Hyperparameter tuning (Optuna)

In [ ]:
trainer_kwargs = {
    # "checkpoint_dir": RESULT_DIR_TRIAL,
    "enable_progress_bar": False,
    "enable_model_summary": False,
    "log_every_n_steps": 5,
    "precision": PRECISION,
    "max_epochs": MAX_EPOCHS,
}

monitor_kwargs = {
    "monitor": MONITOR_METRIC,
    "mode": MONITOR_MODE,
}

early_stopping_kwargs = {
    "patience": 15,
    "verbose": False,
    "monitor": MONITOR_METRIC,
    "mode": MONITOR_MODE,
}


logger_kwargs = {
    "tracking_uri": MLFLOW_TRACKING_URI,
}

technical_settings = {
    "trainer_kwargs": trainer_kwargs,
    "monitor_kwargs": monitor_kwargs,
    "early_stopping_kwargs": early_stopping_kwargs,
    "logger_kwargs": logger_kwargs,
}

In [ ]:
# Per-fold graphs, following medical_data_clean:
#  - baseline "empty" and "fully_connected" graphs (no oracle for real data),
#  - ii_graph_{threshold} from the interaction-information matrix,
#  - node-permuted controls using one reference mapping per permutation seed.
GRAPHS = {}

# Baseline graphs per fold. An empty true-edge list means there is no oracle graph.
for ii in ii_matrices:
    base = build_base_graphs(build_true_graph(ii, []))
    for name, graph in base.items():
        GRAPHS.setdefault(name, []).append(graph)

# Interaction-information graphs and their fixed permutations for each fold and threshold.
for threshold in INTERACTION_THRESHOLDS:
    key = f"ii_graph_{int(threshold * 100)}"
    GRAPHS[key] = []
    permuted = {f"{key}_permuted_{seed}": [] for seed in PERMUTE_SEEDS}
    for ii in ii_matrices:
        fold_graphs = build_ii_graphs(
            ii,
            PERMUTE_SEEDS,
            threshold=threshold,
            permutation_mappings=permutation_mappings,
        )
        GRAPHS[key].append(fold_graphs["ii"])
        for seed in PERMUTE_SEEDS:
            permuted[f"{key}_permuted_{seed}"].append(
                fold_graphs[f"ii_permuted_{seed}"]
            )
    GRAPHS.update(permuted)

In [ ]:
# Audit the full per-fold path: TRAIN-only preprocessing -> II matrix -> graph -> model nodes.
merged_sources = {"tro", "trot"}
merged_output = "troponina"
audit_rows = []
for fold_idx, (fold, ii) in enumerate(zip(folds, ii_matrices)):
    train_index = pd.Index(fold["train"])
    X_fold_train = X.loc[train_index]
    y_fold_train = y.loc[train_index]
    assert X_fold_train.index.equals(y_fold_train.index)

    graph_pipe = copy.deepcopy(GRAPH_PREPROCESSING_PIPELINE)
    graph_input = graph_pipe.fit_transform(X_fold_train, y_fold_train)
    recomputed_ii = compute_info(graph_input, y_fold_train, n_bins=None)

    model_pipe = copy.deepcopy(PREPROCESSING_PIPELINE)
    model_input = model_pipe.fit_transform(X_fold_train, y_fold_train)

    raw_columns = set(X_fold_train.columns)
    expected_columns = (raw_columns - merged_sources) | {merged_output}
    graph_columns = list(graph_input.columns)
    model_columns = list(model_input.columns)

    assert merged_sources <= raw_columns
    assert merged_sources.isdisjoint(graph_columns)
    assert merged_sources.isdisjoint(model_columns)
    assert merged_output in graph_columns
    assert merged_output in model_columns
    assert graph_input.shape[1] == model_input.shape[1] == X_fold_train.shape[1] - 1
    assert set(graph_columns) == set(model_columns) == expected_columns

    assert graph_columns == list(ii.index) == list(ii.columns)
    assert not graph_input.columns.has_duplicates
    assert not model_input.columns.has_duplicates
    assert np.allclose(ii.to_numpy(), recomputed_ii.to_numpy(), equal_nan=True)

    aligned_model_input = model_input[graph_columns]
    assert list(aligned_model_input.columns) == graph_columns

    graph_edge_counts = {}
    graphs_to_check = []
    for threshold in INTERACTION_THRESHOLDS:
        key = f"ii_graph_{int(threshold * 100)}"
        expected_graphs = build_ii_graphs(
            ii,
            PERMUTE_SEEDS,
            threshold=threshold,
            permutation_mappings=permutation_mappings,
        )
        actual_graph = GRAPHS[key][fold_idx]
        assert list(actual_graph.nodes()) == graph_columns
        assert {frozenset(edge) for edge in actual_graph.edges()} == {
            frozenset(edge) for edge in expected_graphs["ii"].edges()
        }
        graph_edge_counts[key] = actual_graph.number_of_edges()
        graphs_to_check.append(actual_graph)

        for seed in PERMUTE_SEEDS:
            permuted_key = f"{key}_permuted_{seed}"
            actual_permuted = GRAPHS[permuted_key][fold_idx]
            expected_permuted = expected_graphs[f"ii_permuted_{seed}"]
            assert set(actual_permuted.nodes()) == set(model_columns)
            assert {frozenset(edge) for edge in actual_permuted.edges()} == {
                frozenset(edge) for edge in expected_permuted.edges()
            }
            graphs_to_check.append(actual_permuted)

    # End-to-end value check used by build_graph_dataset: tensor position i must
    # contain the value of the feature named by graph.nodes[i].
    for graph_to_check in graphs_to_check:
        node_order = list(graph_to_check.nodes())
        sample_frame = model_input.loc[[train_index[0]], node_order]
        sample_target = y_fold_train.loc[[train_index[0]]]
        sample_graph = build_graph_dataset(sample_frame, sample_target, graph_to_check)[
            0
        ]
        assert np.allclose(
            sample_graph.x[:, 0].cpu().numpy(),
            sample_frame.iloc[0].to_numpy(dtype=float),
            equal_nan=True,
        )

    audit_rows.append(
        {
            "fold": fold_idx,
            "train_rows": len(train_index),
            "raw_features": X_fold_train.shape[1],
            "processed_features": graph_input.shape[1],
            "tro_trot_removed": merged_sources.isdisjoint(graph_columns),
            "troponin_added": merged_output in graph_columns,
            "feature_sets_equal": set(graph_columns) == set(model_columns),
            "original_order_equal": graph_columns == model_columns,
            "aligned_to_graph_nodes": list(aligned_model_input.columns)
            == graph_columns,
            "tensor_values_match_node_names": True,
            "ii_recomputed_equal": True,
            "graphs_equal": True,
            "permuted_graphs_equal": True,
            **graph_edge_counts,
        }
    )

fold_graph_audit = pd.DataFrame(audit_rows)
fold_graph_audit

In [ ]:
# Derive the tuning graph selection from INTERACTION_THRESHOLDS and PERMUTE_SEEDS
# so it always matches the graphs built above:
#  - baseline "empty" and "fully_connected" graphs,
#  - one interaction-information graph and its node-permuted negative controls
#    for every threshold.

SELECTED_GRAPH_KEYS = []

if include_empty_full:
    SELECTED_GRAPH_KEYS = SELECTED_GRAPH_KEYS + ["empty", "fully_connected"]

for threshold in INTERACTION_THRESHOLDS:
    key = f"ii_graph_{int(threshold * 100)}"
    SELECTED_GRAPH_KEYS.append(key)
    SELECTED_GRAPH_KEYS += [f"{key}_permuted_{s}" for s in PERMUTE_SEEDS]

In [ ]:
GRAPHS_SELECTED = {k: GRAPHS[k] for k in SELECTED_GRAPH_KEYS}

In [ ]:
graph_diagnostic_rows = []
for graph_name, graph_folds in GRAPHS_SELECTED.items():
    source_name = (
        graph_name[: graph_name.rfind("_permuted_")]
        if "_permuted_" in graph_name
        else None
    )
    for fold_idx, graph in enumerate(graph_folds):
        n_nodes = graph.number_of_nodes()
        n_possible_edges = n_nodes * (n_nodes - 1) // 2
        edge_index, _, _ = prepare_graph(graph)
        if edge_index.numel() == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)

        ii_columns = list(ii_matrices[fold_idx].columns)
        assert set(graph.nodes()) == set(ii_columns)
        assert edge_index.shape[0] == 2
        assert edge_index.shape[1] == 2 * graph.number_of_edges()

        if graph_name == "empty":
            assert graph.number_of_edges() == 0
            assert edge_index.shape == (2, 0)
        elif graph_name == "fully_connected":
            assert graph.number_of_edges() == n_possible_edges

        edge_jaccard_with_source = np.nan
        if source_name is not None:
            source_edges = {
                frozenset(edge)
                for edge in GRAPHS_SELECTED[source_name][fold_idx].edges()
            }
            graph_edges = {frozenset(edge) for edge in graph.edges()}
            edge_union = source_edges | graph_edges
            edge_jaccard_with_source = (
                len(source_edges & graph_edges) / len(edge_union) if edge_union else 1.0
            )

        graph_diagnostic_rows.append(
            {
                "graph_name": graph_name,
                "fold": fold_idx,
                "n_nodes": n_nodes,
                "n_edges": graph.number_of_edges(),
                "edge_density": graph.number_of_edges() / n_possible_edges,
                "directed_edges_in_tensor": edge_index.shape[1],
                "node_order_matches_ii": list(graph.nodes()) == ii_columns,
                "edge_jaccard_with_source": edge_jaccard_with_source,
            }
        )

probe_fold = folds[0]
probe_pipe = copy.deepcopy(PREPROCESSING_PIPELINE)
probe_X = probe_pipe.fit_transform(
    X.loc[probe_fold["train"]],
    y.loc[probe_fold["train"]],
)
probe_columns = list(GRAPHS_SELECTED["empty"][0].nodes())
probe_X = probe_X[probe_columns].iloc[:1]
probe_y = y.loc[probe_fold["train"]].iloc[:1]
empty_sample = build_graph_dataset(probe_X, probe_y, GRAPHS_SELECTED["empty"][0])[0]
full_sample = build_graph_dataset(
    probe_X, probe_y, GRAPHS_SELECTED["fully_connected"][0]
)[0]

torch.manual_seed(SEED)
probe_model = MyGNNConcat(
    n_nodes=len(probe_columns),
    emb_dim=4,
    hidden_dim=8,
    n_layers=1,
    dropout=0.0,
    batch_norm=False,
    heads=1,
    conv_layer="GATConv",
)
probe_model.eval()
probe_batch = torch.zeros(len(probe_columns), dtype=torch.long)
with torch.no_grad():
    empty_logits = probe_model(empty_sample.x, empty_sample.edge_index, probe_batch)
    full_logits = probe_model(full_sample.x, full_sample.edge_index, probe_batch)

logit_max_abs_difference = (empty_logits - full_logits).abs().max().item()
assert np.isfinite(logit_max_abs_difference)
assert logit_max_abs_difference > 1e-7

graph_diagnostics = pd.DataFrame(graph_diagnostic_rows)
print(f"empty-vs-full max |logit difference|: {logit_max_abs_difference:.6g}")
graph_diagnostics

## XGBoost baseline

Cross-validation uses the same folds as GNN/MLP, with 50 TPE trials and early stopping. Runs are logged to the same MLflow experiment (`EXPERIMENT_NAME`) with the same tags (`model_cls=XGBoost`, `graph_name=xgboost_full`, and `DATA_TAGS`). Only training and validation metrics are reported during tuning; the test set remains untouched.

In [ ]:
xgb_result = run_xgboost_fixed_folds_study(
    X=X,
    y=y,
    folds=folds,
    n_trials=N_TRIALS,
    seed=SEED,
    optuna_storage=OPTUNA_STORAGE_URL,
    experiment_name=EXPERIMENT_NAME,
    data_tags=DATA_TAGS,
    mlflow_tracking_uri=MLFLOW_TRACKING_URI,
    n_startup_trials=N_STARTUP_TRIALS,
    xgb_max_rounds=2000,
    xgb_early_stopping_rounds=15,
    xgb_preprocessing_pipeline=copy.deepcopy(XGBOOST_PREPROCESSING_PIPELINE),
    xgb_search_space=medical_cfg.XGB_SEARCH_SPACE,
    run_test=False,
    monitor_metric=MONITOR_METRIC,
)
xgb_result

In [ ]:
results = run_fixed_folds_sweep(
    X=X,
    y=y,
    folds=folds,
    graphs_by_name=GRAPHS_SELECTED,
    model_configs=MODEL_CONFIGS,
    technical_settings=technical_settings,
    direction=DIRECTION,
    n_trials=N_TRIALS,
    sampler_seed=SEED,
    optuna_storage=OPTUNA_STORAGE_URL,
    experiment_name=EXPERIMENT_NAME,
    base_tags=DATA_TAGS,
    run_mlp=True,
    mlp_reference_graph=MLP_REFERENCE_GRAPH,
    standardize=False,
    keep_on_gpu=True,
    n_startup_trials=N_STARTUP_TRIALS,
    pruning_mode=PRUNING_MODE,
    pruning_warmup_steps=PRUNING_WARMUP_STEPS,
    top_k=None,
    main_data_seed=None,
    permuted_top_k=None,
    preprocessing_pipeline=copy.deepcopy(PREPROCESSING_PIPELINE),
    run_test=False,
)

In [ ]:
summary_rows = []

if isinstance(results, list):
    summary_rows.extend(results)

if isinstance(xgb_result, dict):
    summary_rows.append(xgb_result)

summary_df = pd.DataFrame(summary_rows)

preferred_cols = [
    "model_cls",
    "graph_name",
    "best_value",
    "val/auc",
    "val/avg_precision",
    "val/f1",
]
available_cols = [c for c in preferred_cols if c in summary_df.columns]
other_cols = [c for c in summary_df.columns if c not in available_cols]
summary_df = summary_df[available_cols + other_cols]

if "best_value" in summary_df.columns:
    summary_df = summary_df.sort_values("best_value", ascending=False).reset_index(
        drop=True
    )

summary_df

## Next stage

The final refit of the winning models and the single held-out test evaluation are implemented in `final_refit.ipynb`. Run it only after every study in this notebook has completed.